# Vector Search Setup

This notebook creates the Vector Search endpoint, managed embedding indices, and SQL search functions for all configured metadata tables.

**Prerequisites:**
- Metadata tables must exist and be populated
- Unity Catalog must be enabled

In [ ]:
%pip install --upgrade databricks-vectorsearch pydantic --quiet
dbutils.library.restartPython()

In [ ]:
import time
from typing import Literal

from databricks.vector_search.client import VectorSearchClient
from pydantic import BaseModel, computed_field


class VectorSearchIndexConfig(BaseModel):
    """Configuration for a single Vector Search Index."""

    source_table: str
    primary_key: str = "indicator_id"
    embedding_column: str = "embedding_text"
    embedding_model: str = "databricks-bge-large-en"
    pipeline_type: Literal["TRIGGERED", "CONTINUOUS"] = "TRIGGERED"
    sql_function_name: str

    @computed_field
    @property
    def index_name(self) -> str:
        return f"{self.source_table}_index"


class VectorSearchConfig(BaseModel):
    """Top-level configuration for all vector search resources."""

    catalog: str = "main_catalog"
    schema_name: str = "dev"
    endpoint_name: str = "dev_vector_search"
    indices: list[VectorSearchIndexConfig]

    def get_full_table_name(self, table: str) -> str:
        return f"{self.catalog}.{self.schema_name}.{table}"

    def get_full_index_name(self, index_config: VectorSearchIndexConfig) -> str:
        return f"{self.catalog}.{self.schema_name}.{index_config.index_name}"

    def get_full_function_name(self, index_config: VectorSearchIndexConfig) -> str:
        return f"{self.catalog}.{self.schema_name}.{index_config.sql_function_name}"

In [ ]:
# Configuration - add/remove indices here
config = VectorSearchConfig(
    catalog="main_catalog",
    schema_name="dev",
    endpoint_name="worldbank_vector_search", # dev_vector_search 
    indices=[
        VectorSearchIndexConfig(
            source_table="worldbank_indicators",
            sql_function_name="search_worldbank_indicators",
        ),
        VectorSearchIndexConfig(
            source_table="unsdg_indicators",
            sql_function_name="search_unsdg_indicators",
        ),
        VectorSearchIndexConfig(
            source_table="datagov_indicators",
            sql_function_name="search_datagov_indicators",
        ),
        VectorSearchIndexConfig(
            source_table="hdx_indicators",
            sql_function_name="search_hdx_indicators",
        ),
        VectorSearchIndexConfig(
            source_table="cbs_indicators",
            sql_function_name="search_cbs_indicators",
        ),
        VectorSearchIndexConfig(
            source_table="who_indicators",
            sql_function_name="search_who_indicators",
        ),
        # World Bank table search - finds relevant data tables by description
        VectorSearchIndexConfig(
            source_table="worldbank_tables",
            primary_key="table_name",  # Different primary key
            sql_function_name="search_worldbank_tables",
        ),
    ],
)

client = VectorSearchClient()

print(f"Catalog: {config.catalog}")
print(f"Schema: {config.schema_name}")
print(f"Endpoint: {config.endpoint_name}")
print(f"Indices to create: {len(config.indices)}")
for idx in config.indices:
    print(f"  - {idx.source_table} -> {idx.index_name}")

## Step 1: Create/Verify Vector Search Endpoint

In [ ]:
def create_endpoint_if_not_exists(client: VectorSearchClient, endpoint_name: str) -> None:
    """Create the vector search endpoint if it doesn't exist."""
    try:
        endpoint = client.get_endpoint(endpoint_name)
        print(f"Endpoint '{endpoint_name}' already exists")
    except Exception as e:
        if "RESOURCE_DOES_NOT_EXIST" in str(e) or "NOT_FOUND" in str(e):
            print(f"Creating endpoint '{endpoint_name}'...")
            client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
            print(f"Endpoint '{endpoint_name}' created.")
        else:
            raise e


def wait_for_endpoint(client: VectorSearchClient, endpoint_name: str) -> None:
    """Wait for endpoint to be online."""
    while True:
        endpoint = client.get_endpoint(endpoint_name)
        status = endpoint.get("endpoint_status", {}).get("state", "UNKNOWN")
        print(f"Endpoint status: {status}")
        if status == "ONLINE":
            print("Endpoint is ready!")
            break
        elif status in ["PROVISIONING", "PENDING"]:
            print("Waiting 30 seconds...")
            time.sleep(30)
        else:
            print(f"Unexpected status: {status}")
            break


create_endpoint_if_not_exists(client, config.endpoint_name)
wait_for_endpoint(client, config.endpoint_name)

## Step 2: Enable Change Data Feed on Source Tables

In [ ]:
for index_config in config.indices:
    source_table = config.get_full_table_name(index_config.source_table)
    spark.sql(f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"Change Data Feed enabled on {source_table}")

## Step 3: Create Managed Embedding Indices

In [ ]:
def create_index_if_not_exists(
    client: VectorSearchClient,
    endpoint_name: str,
    index_name: str,
    source_table: str,
    primary_key: str,
    embedding_column: str,
    embedding_model: str,
    pipeline_type: str,
) -> None:
    """Create a vector search index if it doesn't exist."""
    try:
        index = client.get_index(endpoint_name=endpoint_name, index_name=index_name)
        print(f"Index '{index_name}' already exists")
    except Exception as e:
        if "RESOURCE_DOES_NOT_EXIST" in str(e) or "NOT_FOUND" in str(e):
            print(f"Creating managed embedding index '{index_name}'...")
            client.create_delta_sync_index(
                endpoint_name=endpoint_name,
                index_name=index_name,
                source_table_name=source_table,
                primary_key=primary_key,
                embedding_source_column=embedding_column,
                embedding_model_endpoint_name=embedding_model,
                pipeline_type=pipeline_type,
            )
            print(f"Index '{index_name}' created and syncing...")
        else:
            raise e


for index_config in config.indices:
    create_index_if_not_exists(
        client=client,
        endpoint_name=config.endpoint_name,
        index_name=config.get_full_index_name(index_config),
        source_table=config.get_full_table_name(index_config.source_table),
        primary_key=index_config.primary_key,
        embedding_column=index_config.embedding_column,
        embedding_model=index_config.embedding_model,
        pipeline_type=index_config.pipeline_type,
    )
    print()

## Step 4: Check Index Sync Status

In [ ]:
for index_config in config.indices:
    index_name = config.get_full_index_name(index_config)
    index = client.get_index(endpoint_name=config.endpoint_name, index_name=index_name)
    status = index.describe().get("status", {})
    print(f"{index_config.source_table}: {status.get('ready', 'UNKNOWN')} - {status.get('message', '')}")

## Step 5: Create SQL Search Functions

In [ ]:
for index_config in config.indices:
    function_name = config.get_full_function_name(index_config)
    index_name = config.get_full_index_name(index_config)

    # Note: num_results must be a literal constant (foldable expression) in VECTOR_SEARCH
    # It cannot be passed as a dynamic parameter
    create_function_sql = f"""
    CREATE OR REPLACE FUNCTION {function_name}(query STRING)
    RETURNS TABLE
    RETURN SELECT * FROM VECTOR_SEARCH(
        index => '{index_name}',
        query => query,
        num_results => 10
    )
    """
    spark.sql(create_function_sql)
    print(f"Created function: {function_name}")

## Step 6: Test the Search Functions

In [ ]:
test_query = "poverty and inequality measures"

for index_config in config.indices:
    function_name = config.get_full_function_name(index_config)
    print(f"\n=== Testing {function_name} ===")
    try:
        results = spark.sql(f"SELECT * FROM {function_name}('{test_query}')")
        display(results)
    except Exception as e:
        print(f"Error (index may still be syncing): {e}")

## Manual Index Sync

To refresh indices after source table updates:

In [ ]:
def sync_all_indices(client: VectorSearchClient, config) -> None:
    """Trigger sync on all configured indices."""
    for index_config in config.indices:
        index_name = config.get_full_index_name(index_config)
        index = client.get_index(endpoint_name=config.endpoint_name, index_name=index_name)
        index.sync()
        print(f"Sync triggered for {index_name}")


# Uncomment to sync all indices:
# sync_all_indices(client, config)

## Experimental: Query with Reranking

Use this section to experiment with different query options including hybrid search and reranking.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from databricks.vector_search.reranker import DatabricksReranker

# Configuration - modify these to experiment
ENDPOINT_NAME = "worldbank_vector_search"  # Change to your endpoint
INDEX_NAME = "main_catalog.dev.worldbank_indicators_index"  # Change to your index
QUERY = "poverty and inequality measures"
NUM_RESULTS = 10

# Initialize client and get index
client = VectorSearchClient()
index = client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

# Columns to retrieve
columns = ["indicator_id", "indicator_name", "long_definition", "embedding_text"]

# --- Option 1: Standard similarity search ---
print("=== Standard Similarity Search ===")
results_standard = index.similarity_search(
    query_text=QUERY,
    columns=columns,
    num_results=NUM_RESULTS,
)
print(f"Found {len(results_standard.get('result', {}).get('data_array', []))} results")

# --- Option 2: Hybrid search (vector + keyword) ---
print("\n=== Hybrid Search ===")
results_hybrid = index.similarity_search(
    query_text=QUERY,
    columns=columns,
    num_results=NUM_RESULTS,
    query_type="hybrid",
)
print(f"Found {len(results_hybrid.get('result', {}).get('data_array', []))} results")

# --- Option 3: Hybrid search with reranking ---
print("\n=== Hybrid Search with Reranking ===")
results_reranked = index.similarity_search(
    query_text=QUERY,
    columns=columns,
    num_results=NUM_RESULTS,
    query_type="hybrid",
    reranker=DatabricksReranker(
        columns_to_rerank=["indicator_name", "long_definition"]
    ),
)
print(f"Found {len(results_reranked.get('result', {}).get('data_array', []))} results")

In [ ]:
# Compare results from different query methods
import pandas as pd

def results_to_df(results, method_name):
    """Convert vector search results to a DataFrame."""
    data = results.get("result", {}).get("data_array", [])
    col_names = [c["name"] for c in results.get("manifest", {}).get("columns", [])]
    df = pd.DataFrame(data, columns=col_names)
    df["method"] = method_name
    df["rank"] = range(1, len(df) + 1)
    return df

df_standard = results_to_df(results_standard, "standard")
df_hybrid = results_to_df(results_hybrid, "hybrid")
df_reranked = results_to_df(results_reranked, "reranked")

# Show top 5 from each method side by side
print("=== Top 5 Results Comparison ===\n")
for method, df in [("Standard", df_standard), ("Hybrid", df_hybrid), ("Reranked", df_reranked)]:
    print(f"--- {method} ---")
    for _, row in df.head(5).iterrows():
        print(f"  {row['rank']}. {row['indicator_name'][:60]}...")
    print()

In [ ]:
# Example with filters (uncomment and modify as needed)
# Filters use SQL-like syntax: =, !=, <, <=, >, >=, IN, LIKE, OR

# results_filtered = index.similarity_search(
#     query_text=QUERY,
#     columns=columns,
#     num_results=NUM_RESULTS,
#     query_type="hybrid",
#     filters="source LIKE '%World Bank%'",
#     reranker=DatabricksReranker(
#         columns_to_rerank=["indicator_name", "long_definition"]
#     ),
# )
# print(f"Filtered results: {len(results_filtered.get('result', {}).get('data_array', []))}")